[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P03/P03-2_Etude_Bielle.ipynb)

# Étude I : bielle en traction

Étude de synthèse de la partie I du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

Cette étude reprend, sur un cas complet, ce que les chapitres 2 et 3 ont mis en place : une formule et ses unités, des listes et des boucles, une fonction testée, une dichotomie. Une bielle de vérin, en acier de limite d'élasticité $R_e = 355$ MPa, transmet un effort de traction. Le cahier des charges demande un coefficient de sécurité d'au moins 2,0, c'est-à-dire une contrainte de service d'au plus la moitié de la limite d'élasticité. Trois bielles sont à dimensionner, pour des efforts de 45, 60 et 80 kN, chacune avec le plus petit diamètre de la série normalisée qui convient. Le résultat attendu est un diamètre par bielle, sa sécurité vérifiée, et une fonction qui refuse les appels absurdes. Aucun jeu de données : tout part de l'énoncé. La figure du chapitre donne la feuille de route.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Fonctions du module du livre (`meca.py`)

Le code ci-dessous reprend, tel qu'il est écrit dans le livre, les fonctions du module employées dans ce notebook ; ce code est fourni tel quel.

In [ ]:
def dichotomie(g, a, b, tolerance=0.005):
    """Zéro de g dans [a, b] par dichotomie : rend (solution, nombre de tours).

    g doit changer de signe entre a et b. La boucle s'arrête quand l'encadrement fait moins que la tolérance.

    >>> x, tours = dichotomie(lambda d: tau_MPa(d, 500.0) - 60.0, 10.0, 100.0)
    >>> round(x, 2), tours
    (34.88, 15)
    >>> dichotomie(lambda x: x**2 + 1.0, 0.0, 1.0)
    Traceback (most recent call last):
        ...
    ValueError: g ne change pas de signe entre a et b
    """
    if g(a) * g(b) > 0:
        raise ValueError("g ne change pas de signe entre a et b")
    tours = 0
    while b - a > tolerance:
        milieu = (a + b) / 2
        a_garde = g(a) * g(milieu) <= 0      # le signe change à gauche
        a, b = (a, milieu) if a_garde else (milieu, b)
        tours = tours + 1
    return (a + b) / 2, tours

### Préparation

In [ ]:
import math

def contrainte_MPa(F_N, d_mm):
    """Contrainte normale dans une tige ronde, en MPa."""
    if d_mm <= 0:
        raise ValueError("le diamètre doit être positif")
    return F_N / (math.pi * d_mm**2 / 4)

def securite(Re_MPa, F_N, d_mm):
    """Coefficient de sécurité : limite d'élasticité sur contrainte de service."""
    return Re_MPa / contrainte_MPa(F_N, d_mm)

## Étape 1 : contrainte et sécurité

Les données du problème et la contrainte de la bielle de 20 mm.

Les contrôles de la cellule lisent la variable `sigma_20` : gardez ce nom.

In [ ]:
Re_MPa = 355.0
s_requis = 2.0
efforts_kN = {"bielle A": 45.0, "bielle B": 60.0, "bielle C": 80.0}
# Votre code :
assert abs(sigma_20 - 143.2) < 0.05, "contrainte attendue : 143,2 MPa"
assert abs(securite(Re_MPa, 45000.0, 20.0) - 2.48) < 0.005, "sécurité attendue : 2,48"
print("Étape 1 : contrôles passés")

## Étape 2 : diamètre minimal

Le diamètre minimal continu, par la formule fermée puis par dichotomie.

Les contrôles de la cellule lisent les variables `d_dicho`, `d_formule`, `d_module`, `tours`, `tours_module` : gardez ces noms.

In [ ]:
# Votre code :
assert abs(d_formule - 17.97) < 0.005, "diamètre minimal attendu : 17,97 mm"
assert abs(d_dicho - d_formule) < 0.005, "la dichotomie retrouve la formule"
# Votre code :
assert d_module == d_dicho and tours_module == tours, "la fonction du module refait la boucle"
print("Étape 2 : contrôles passés")

## Étape 3 : diamètre du catalogue

La série normalisée, parcourue en boucle ; on garde le premier diamètre qui convient.

Les contrôles de la cellule lisent la variable `choix` : gardez ce nom.

In [ ]:
serie_mm = [10, 12, 14, 16, 18, 20, 22, 25, 28, 30, 32, 36, 40]
# Votre code :
assert (
    choix["bielle A"][0] == 18 and abs(choix["bielle A"][1] - 2.0075) < 0.0005
), "bielle A : 18 mm, s = 2,0075"
assert choix["bielle B"][0] == 22 and choix["bielle C"][0] == 25, "bielles B et C : 22 mm et 25 mm"
print("Étape 3 : contrôles passés")

## Étape 4 : tests et garde-fous

Tests et garde-fous.

Les contrôles de la cellule lisent les variables `d_A`, `masse_kg`, `messages` : gardez ces noms.

In [ ]:
# Votre code :
assert securite(Re_MPa, 45000.0, d_A) >= s_requis, "le diamètre retenu tient la sécurité demandée"
assert d_A >= math.ceil(d_formule), "jamais sous le minimum continu"
assert diametre_normalise(355.0, 45000.0, 2.0, [16, 18, 20]) == 18, "cas de référence : 18 mm"
# Votre code :
assert messages == ["l'effort doit être positif", "aucun diamètre de la série ne convient"]
# Votre code :
assert abs(masse_kg - 0.80) < 0.005, "masse attendue : 0,80 kg"
# Votre code :
print("Étape 4 : contrôles passés")